# 22 · Full-precision export of the per-image results (CPU, about 5 min)
**Runtime: CPU.** No training, no model is loaded. Reads the per-run evaluation files that notebooks 14 and 20 wrote (full precision)
and writes the same per-image table as `eval_slim.csv.gz` / `eval_dmid_v2_slim.csv.gz`, but **without rounding**, split into small files.
Writes only to `results/p2_full_precision/`. Nothing else is changed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, numpy as np, pandas as pd
PROJECT = '/content/drive/MyDrive/quantum-placement'
EV1 = f'{PROJECT}/results/p2_jit_eval/eval_per_image_all.csv'       # CBIS-DDSM + DMID (original rule), notebook 14
EV2 = f'{PROJECT}/results/p2_dmid_v2/eval'                         # DMID (corrected rule), notebook 20, one csv per run
OUT = f'{PROJECT}/results/p2_full_precision'; os.makedirs(OUT, exist_ok=True)
COLS = ['set', 'model', 'fold', 'seed', 'dataset', 'roi_id', 'keep', 'dice', 'iou', 'hd95', 'dice_const', 'flip_const', 'absdp_const',
        'flip_swap', 'absdp_swap'] + [f'dice_swap{k}' for k in range(10)] + ['epochs_run', 'best_epoch']
a = pd.read_csv(EV1)
b = pd.concat([pd.read_csv(f) for s in ('train', 'val') for f in sorted(glob.glob(f'{EV2}/{s}/*.csv'))], ignore_index=True)
for t in (a, b):
    for c in COLS:
        if c not in t: t[c] = np.nan
b['dataset'] = 'DMID'
if b['keep'].isna().all(): b['keep'] = True
print('notebook 14 rows:', len(a), a.groupby(['dataset', 'set']).size().to_dict())
print('notebook 20 rows:', len(b), b.groupby('set').size().to_dict(), '| runs:', b.groupby(['set', 'model', 'fold', 'seed']).ngroups)

In [ ]:
# Write unrounded files (float repr keeps full float64 precision), one file per dataset x schedule
def save(t, name):
    p = f'{OUT}/{name}.csv.gz'; t[COLS].to_csv(p, index=False, compression='gzip', float_format='%.17g'); print(name, len(t), 'rows,', round(os.path.getsize(p) / 1e6, 2), 'MB')
cb = a[a.dataset == 'CBIS']; d1 = a[a.dataset == 'DMID']
for s in ('train', 'val'):
    save(cb[cb.set == s], f'full_cbis_{s}')
    save(b[b.set == s], f'full_dmid_corrected_{s}')
    save(d1[d1.set == s], f'full_dmid_original_rule_{s}')

In [ ]:
# Check against the 4-decimal files: every value must differ by at most 0.00005
def chk(full, slim_path, label):
    if not os.path.exists(slim_path): print(label, ': slim file not found, skipped'); return
    s = pd.read_csv(slim_path); k = ['set', 'model', 'fold', 'seed', 'dataset', 'roi_id']
    m = full[COLS].merge(s, on=k, suffixes=('', '_s'))
    num = [c for c in COLS if c not in k + ['keep', 'epochs_run', 'best_epoch']]
    d = max(float(np.nanmax(np.abs(m[c] - m[c + '_s']))) if m[c].notna().any() else 0.0 for c in num)
    print(label, '| matched rows:', len(m), 'of', len(full), '| max |full - 4dp|:', d)
chk(a, f'{PROJECT}/results/p2_jit_eval/eval_slim.csv.gz', 'notebook 14')
chk(b, f'{PROJECT}/results/p2_dmid_v2/eval_dmid_v2_slim.csv.gz', 'notebook 20')
print(sorted(os.listdir(OUT)))
print('DONE')